## Project starter

### This is intended to be a starting notebook for a project using the QM9 dataset
 - Molecules consist of H,C,O,N and F atoms, and contain up to 9 heavy atoms.
 - More information is available here: https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.QM9.html#torch_geometric.datasets.QM9

In [ ]:
!pip install torch
!pip install torch_geometric
!pip install rdkit

In [5]:
import torch
from torch_geometric.transforms import BaseTransform, Compose

from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv
import torch.nn.functional as f
from torch.nn import Linear, ReLU

from torch_geometric.nn import global_mean_pool
from matplotlib import pyplot as plt

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('using', device)


using cpu


## Some tools to help with the data

In [5]:

class SetTarget(BaseTransform):
    '''
    Transform to modify target vector so there is a single value.

    Choose from 0-18 for target values described here:
    https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.QM9.html#torch_geometric.datasets.QM9
    '''
    def __init__(self,target):
        self.target = target

    def __call__(self,data):
        data.y = data.y[:,self.target]
        return data

class NormalizeTarget(BaseTransform):
    '''
    Transform to normalize target vector.

    Parameters:
    ----------
    mean : mean of the training data target values
    std : standard deviation of the training data target values

    '''
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, data):
        data.y = (data.y - self.mean) / self.std
        return data

## Plotting methods

In [6]:
# Some functions to plot stuff

def plot_pred_v_actual(preds, actual, title=''):
    '''
    Plot predicted vs actual/expected values as a scatter plot

    Parameters:
    ----------
    preds : numpy array
            predicted values
    actual : numpy array
            expected values
    title : string
            Title for plot (optional)
    '''
    fig, ax = plt.subplots(figsize=(10,7.5))

    # Set the default text font size
    plt.rc('figure', titlesize=16)
    plt.rc('font', size=12)

    min_val = min(min(preds), min(actual))
    max_val = max(max(preds), max(actual))
    lims = [min_val, max_val]

    # now plot both limits against eachother
    ax.plot(lims, lims, 'k--', alpha=0.75)

    ax.set_aspect('equal')
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    ax.scatter(actual, preds, s=20)

    if title == '':
        ax.set_title("Predicted vs Actuals")
    else:
        ax.set_title(title)

    plt.xlabel("actual")
    plt.ylabel("predicted")

    plt.show()


def plot_losses(tlosses, vlosses):
    '''
    Plot training and validation losses per epoch
    Expectation is that losses are ordered starting with same epoch (0)

    Parameters:
    ----------
    tlosses : list of floats
                training loss values
    vlosses : list of floats
                validation loss values

    '''
    plt.figure(figsize=(10,5))
    plt.plot(tlosses, label = "Training", c='r')
    plt.plot(vlosses, label = "Validation", c='b')
    plt.legend()
    plt.ylabel("Loss")
    plt.xlabel("Epoch")
    plt.title("Training and Validation Loss")
    plt.show()


## Loading the dataset

In [7]:
# this will create a folder called data in the same directory as this notebook
qm9 = QM9(root='data')


Extracting data/raw/qm9.zip
Processing...
100%|██████████| 133885/133885 [02:40<00:00, 833.21it/s] 
Done!


## Creating train/val/test datasets

When we created our samples, we only used part of the dataset (20k), but you may want to increase that for your project.
We also used target 11, so trying a different target would be a good idea. You can change that in the cell below.

In [8]:

#shuffle first so we get a nice variety
qm9 = qm9.shuffle()
#qm9.data.to(device)

# let's use 20k, and split to train/val/test 70/10/20
sample_count = 20000
train_end = int(sample_count * 0.7)
val_end = train_end + int(sample_count* 0.1)
test_end = val_end + int(sample_count * 0.2)

# normalizing the target values can help when using neural networks, let's do that
# collect the target y values from the training to use for normalizing
train_y2 =torch.cat([qm9[i].y for i in range(train_end)], dim=0)

# compute mean and standard deviation
train_mean = train_y2.mean()
train_std = train_y2.std()

# We are going set our prediction target variable here and also normalize in the transform:
# 11 = Heat capacity at 298.15K
target_index = 11

# transform values
qm9.transform = Compose([SetTarget(target_index),NormalizeTarget(train_mean, train_std)])


# use the DataLoader
# set the batch size
batch = 50
# here we need a cpu device for the generator or collab will complain
gen = torch.Generator(device='cpu')
train_loader = DataLoader(qm9[:train_end], batch_size = batch, shuffle = True, generator=gen)
val_loader = DataLoader(qm9[train_end:val_end], batch_size = batch, shuffle = True, generator=gen)
test_loader = DataLoader(qm9[val_end:test_end], batch_size = batch, shuffle = False, generator=gen)



## Some Models

### Simple GCN

We're leaving this here as a baseline, but you'll want to use this general structure to create your own model. 

In [9]:
# Starting with a simple GCN

class GCN(torch.nn.Module):
    '''
    Graph Convolutional Network

    '''
    def __init__(self,num_features, h):
        '''
        Init for GCN class

        Parameters:
        num_features : integer
                     number of features from the data
        h : integer
            number of hidden layers

        '''
        super().__init__()
        self.conv1 = GCNConv(num_features, h)
        self.conv2 = GCNConv(h, h)
        self.linear = Linear(h, 1)

    def forward(self, data):
        e = data.edge_index
        x = data.x

        x = self.conv1(x, e)
        x = x.relu()
        x = self.conv2(x, e)
        x = global_mean_pool(x, data.batch)

        x = f.dropout(x, p=0.25, training=self.training)
        x = self.linear(x)

        return x


### Methods for Training and Evaluating the Models

In [9]:
# This is our training method, notice it's not model specific, so we will use it with any model we want to try

# Here we are printing the loss values for train and validation every 50 epochs, but we could modify this to
# print more or less often

def train_model(model, epochs, loss_function, optimizer, train_loader, val_loader):
    '''
    Train the model

    Parameters:
    ----------
    model : nn model to train

    epochs : integer
            number of epochs to train the model

    loss_function : function to evaluate model performance

    optimizer : optimizer to use during training

    train_loader : dataloader containing training data

    val_loader : dataloader containing validation data

    device : where to run

    Returns:
    -------
    model : trained model

    tlosses: list containing floats
            training losses from each epoch

    vlosses : list containing floats
            validation losses from each epoch
    '''
    tlosses = []
    vlosses = []

    for epoch in range(epochs):
        # need model in training mode
        model.train()
        device = next(model.parameters()).device
        tloss = 0
        for data in train_loader:
            #data.x = data.x.float()
            data = data.to(device)
            # forward step of model training
            out = model(data)
            loss = loss_function(out, torch.reshape(data.y, (len(data.y), 1)))
            tloss += loss/len(train_loader)
            # clear the gradients
            optimizer.zero_grad()
            # derive gradients
            loss.backward()
            # update parameters based on gradients
            optimizer.step()

        tlosses.append(tloss.item())

        # check the validation loss after this training epoch
        model.eval()
        with torch.no_grad():
            vloss = 0
            for data in val_loader:
                data = data.to(device)
                out = model(data)
                vloss += loss_function(out, torch.reshape(data.y, (len(data.y),1)))/len(val_loader)

            vlosses.append(vloss.item())

            if epoch%50 == 0:
                print(f"Epoch {epoch}: training loss:{tloss:.6e}, validation loss:{vloss:.6e}")
    print(f"Epoch {epoch}: training loss:{tloss:.6e}, validation loss:{vloss:.6e}")

    return model, tlosses, vlosses

In [10]:
# Here we are using @torch.no_grad() because we don't need to calculate gradients while doing evaluation in this function
# disabling this will improve performance (reduces memory usage) this is the same effect as using with torch.no_grad() as we did above
# also notice here we are returning only the predictions, since the model is not changing in the function, returning it is unnecessary

@torch.no_grad()
def evaluate_test(model, test_loader):
    '''
    Evaluate the test data

    Parameters:
    ----------
    model : trained model

    test_loader : data to evaluate

    Returns:
    -------
    Predicted and expected values
    '''
    model.eval()
    device = next(model.parameters()).device

    first = True

    for data in test_loader:
        data.to(device)
        test_out = model(data)
        test_out = torch.squeeze(test_out, 1)

        if first:
            ttest_out = test_out.detach()
            ttest_y = data.y.detach()
            first = False
        else:
            ttest_out = torch.concat((ttest_out, test_out.detach()))
            ttest_y = torch.concat((ttest_y, data.y.detach()))

    print(f"{loss_function(ttest_out, ttest_y).item():.6e}")

    return ttest_out, ttest_y

## Run the GCN model and plot the results 
- this is your baseline model to compare your results against. Hopefully your model will do better than this one, but remember that it's more important to understand what happens when you make changes than to create the world's best model here. And learning the effect of your changes will help you build a better model. Win-win!

In [ ]:
model = GCN(qm9.num_features, h = 64)
model.to(device)

epochs = 200
loss_function = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

model, tlosses, vlosses = train_model(model, epochs, loss_function, optimizer, train_loader, val_loader)

test_out, test_y = evaluate_test(model, test_loader)

# denormalize
denorm_out = (test_out * train_std) + train_mean
denorm_y = (test_y * train_std) + train_mean
print(f"Test loss:{loss_function(denorm_out, denorm_y).item():.6e}")

preds = denorm_out.cpu().detach().numpy()
actual = denorm_y.cpu().detach().numpy()

plot_pred_v_actual(preds, actual)

### Some modifications you might consider:

 - Change the number of outputs from each layer
 - Adding more convolutional layers
 - Increase the number of outputs of an earlier layer and then reduce
 - Keep the number of outputs the same throughout
 - Reduce the number of outputs earlier
 - Anything else you want to try...

### Dropout layers
Adding dropout layers can help make a model more robust. We will add one to see if it helps here.

&rarr; You can also modify the dropout layer's parameters. Make adjustments and see what they do.

## More Pytorch Geometric Models to Explore
There are MANY more convolutional layers and completed models readily available in PyG. Check them out here and try creating your own combination of layers.

&rarr; https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html

Also try different pooling options and compare the effects.

This tutorial just introduces the QM9 dataset usage, but there are **MANY** other collabs you can check out:

&rarr; https://pytorch-geometric.readthedocs.io/en/latest/get_started/colabs.html